# Detección Inteligente de Anomalías en Tráfico de Red

## Introducción

La detección automática de anomalías es fundamental en sistemas de seguridad modernos. 
Este cuaderno implementa dos enfoques complementarios:

1. **Machine Learning Clásico** — Isolation Forest para detección no supervisada
2. **Deep Learning** — Autoencoder neuronal para modelado de normalidad

Ambos métodos identifican patrones de tráfico inusuales que pueden indicar:
- Intentos de exfiltración de datos
- Conexiones a servidores maliciosos
- Comportamientos anómalos de sistemas
- Amplificación de ataques (botnet activity)

## Generación del Dataset Sintético de Tráfico Red

Creamos un dataset simulado con tráfico normal y anómalo para desarrollar y validar
los modelos de detección. En producción, estos se reemplazarían por datos reales capturados
de routers o firewalls empresariales.

> **Nota importante:** Esta celda genera el dataset que será utilizado por los notebooks
> posteriores. Ejecutar primero este notebook es requisito para los demás.


In [ ]:
import numpy as np
import pandas as pd
import os

# Fijamos la semilla para reproducibilidad
np.random.seed(42)

# Parámetros del dataset
n_normal_samples = 5000    # conexiones de tráfico normal
n_anomaly_samples = 250    # conexiones anómalas (~5%)

# ─────────────────────────────────────────────────────────────────────
# TRÁFICO NORMAL: Características de conexiones legítimas
# ─────────────────────────────────────────────────────────────────────
normal_data = pd.DataFrame({
    'bytes_sent':  np.random.exponential(500, n_normal_samples).astype(int),
    'bytes_recv':  np.random.exponential(800, n_normal_samples).astype(int),
    'duration':    np.random.exponential(30, n_normal_samples).round(2),
    'protocol':    np.random.choice(['TCP', 'UDP', 'ICMP'], n_normal_samples, p=[0.7, 0.25, 0.05]),
    'src_port':    np.random.randint(1024, 65535, n_normal_samples),
    'dst_port':    np.random.choice([80, 443, 53, 22, 8080], n_normal_samples),
    'label':       0   # Etiqueta: 0 = normal
})

# ─────────────────────────────────────────────────────────────────────
# TRÁFICO ANÓMALO: Características de conexiones sospechosas
# ─────────────────────────────────────────────────────────────────────
anomaly_data = pd.DataFrame({
    'bytes_sent':  np.random.exponential(5000, n_anomaly_samples).astype(int),
    'bytes_recv':  np.random.exponential(50, n_anomaly_samples).astype(int),
    'duration':    np.random.exponential(200, n_anomaly_samples).round(2),
    'protocol':    np.random.choice(['TCP', 'UDP', 'ICMP'], n_anomaly_samples, p=[0.3, 0.3, 0.4]),
    'src_port':    np.random.randint(1, 1024, n_anomaly_samples),
    'dst_port':    np.random.choice([4444, 31337, 6667, 9999], n_anomaly_samples),
    'label':       1   # Etiqueta: 1 = anómalo
})

# Combinar datasets y mezclar
dataset = pd.concat([normal_data, anomaly_data], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

# Crear directorio y guardar
os.makedirs('data', exist_ok=True)
dataset.to_csv('data/network_traffic.csv', index=False)

print(f"[+] Dataset guardado: data/network_traffic.csv")
print(f"    Total de registros: {len(dataset)}")
print(f"    - Normales: {(dataset.label == 0).sum()}")
print(f"    - Anómalos: {(dataset.label == 1).sum()}")

---

## Carga y Exploración del Dataset

### Análisis exploratorio inicial del tráfico de red


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar el dataset de tráfico de red
df = pd.read_csv('data/network_traffic.csv')

# ─ Resumen básico ─────────────────────────────────────────────────
print("=" * 60)
print("EXPLORACIÓN DEL DATASET DE TRÁFICO DE RED")
print("=" * 60)
print("\nDimensiones:")
print(f"  Filas: {df.shape[0]}")
print(f"  Columnas: {df.shape[1]}")

print("\nTipos de datos:")
print(df.dtypes)

print("\nPrimeras muestras:")
print(df.head(8))

print("\nEstadísticas descriptivas:")
print(df.describe())

print("\nValores faltantes:")
print(df.isnull().sum())

print(f"\nDistribución de clases:")
print(f"  Normal (0): {(df.label==0).sum()}")
print(f"  Anómalo (1): {(df.label==1).sum()}")

### Preparación de características: Normalización y codificación


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar datos
df = pd.read_csv('data/network_traffic.csv')

# 1. Limpieza: eliminar filas con valores faltantes
df_clean = df.dropna()

# 2. Codificación de variables categóricas (protocolo)
encoder = LabelEncoder()
if df_clean['protocol'].dtype == object:
    df_clean['protocol'] = encoder.fit_transform(df_clean['protocol'])
    print(f"Protocolo codificado: {dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))}")

# 3. Seleccionar características numéricas relevantes para detección
feature_columns = ['bytes_sent', 'bytes_recv', 'duration', 'src_port', 'dst_port', 'protocol']
X = df_clean[feature_columns].copy()
y = df_clean['label'].copy()

# 4. Escalado a [0, 1] usando MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_columns)

print(f"\n✓ Datos preparados:")
print(f"  - Shape: {X_scaled.shape}")
print(f"  - Rango original bytes_sent: [{X['bytes_sent'].min():.0f}, {X['bytes_sent'].max():.0f}]")
print(f"  - Rango escalado bytes_sent: [{X_scaled_df['bytes_sent'].min():.4f}, {X_scaled_df['bytes_sent'].max():.4f}]")

# 5. Visualizar distribuciones antes/después escalado
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for idx, col in enumerate(['bytes_sent', 'bytes_recv', 'duration']):
    # Antes
    axes[0, idx].hist(X[col], bins=40, color='steelblue', alpha=0.7, edgecolor='black')
    axes[0, idx].set_title(f'{col} (Original)')
    axes[0, idx].set_ylabel('Frecuencia')
    
    # Después
    axes[1, idx].hist(X_scaled_df[col], bins=40, color='orange', alpha=0.7, edgecolor='black')
    axes[1, idx].set_title(f'{col} (Escalado)')
    axes[1, idx].set_ylabel('Frecuencia')

plt.suptitle('Distribuciones: Antes vs Después de Normalización', fontsize=12, y=0.995)
plt.tight_layout()
plt.savefig('data/preprocessing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico guardado: data/preprocessing_comparison.png")

---

## Modelo 1: Isolation Forest - Partición Recursiva

El algoritmo de **Isolation Forest** genera aislamientos de puntos mediante particiones aleatorias.
Los puntos anómalos requieren pocas particiones para quedar aislados (alto "anomaly score"),
mientras que los normales necesitan más divisiones.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ─ Preparar datos ─────────────────────────────────────────────────────
df = pd.read_csv('data/network_traffic.csv').dropna()

# Codificar protocolo
le = LabelEncoder()
if df['protocol'].dtype == object:
    df['protocol'] = le.fit_transform(df['protocol'])

feature_cols = ['bytes_sent', 'bytes_recv', 'duration', 'src_port', 'dst_port', 'protocol']
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df[feature_cols])

# ─ Entrenar Isolation Forest ──────────────────────────────────────────
print("=" * 60)
print("ISOLATION FOREST - DETECCIÓN DE ANOMALÍAS")
print("=" * 60)

iso_forest = IsolationForest(
    n_estimators=200,       # número de árboles de aislamiento
    contamination=0.05,     # esperamos ~5% de anomalías
    random_state=42,
    n_jobs=-1               # usar todos los procesadores disponibles
)
iso_forest.fit(X_scaled)

# ─ Predicción y puntuación ────────────────────────────────────────────
predictions = iso_forest.predict(X_scaled)     # +1 = normal, -1 = anomalía
anomaly_scores = iso_forest.decision_function(X_scaled)  # puntuación (menor = más anómalo)

# ─ Agregar resultados al dataframe ────────────────────────────────────
df['IF_prediction'] = predictions
df['IF_anomaly_score'] = anomaly_scores

# ─ Resumen de detecciones ─────────────────────────────────────────────
n_anomalies = (predictions == -1).sum()
print(f"\n[Resultados Isolation Forest]")
print(f"  Total de muestras analizadas: {len(predictions)}")
print(f"  Anomalías detectadas: {n_anomalies} ({100*n_anomalies/len(predictions):.2f}%)")
print(f"  Muestras normales: {(predictions == 1).sum()}")

# Top 15 anomalías más severas
anomalous_df = df[df['IF_prediction'] == -1].sort_values('IF_anomaly_score')
print(f"\nTop 15 anomalías detectadas:")
print(anomalous_df[['bytes_sent', 'bytes_recv', 'duration', 'IF_anomaly_score']].head(15))

# ─ Evaluación contra etiquetas reales ─────────────────────────────────
if 'label' in df.columns:
    y_pred_bin = (predictions == -1).astype(int)
    y_true = df['label'].values
    
    print("\n" + "=" * 60)
    print("EVALUACIÓN CONTRA ETIQUETAS REALES")
    print("=" * 60)
    print(classification_report(y_true, y_pred_bin, target_names=['Normal', 'Anomalía']))
    
    # Matriz de confusión
    cm = confusion_matrix(y_true, y_pred_bin)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Normal', 'Anomalía'], 
                yticklabels=['Normal', 'Anomalía'])
    plt.title('Matriz de Confusión - Isolation Forest')
    plt.ylabel('Etiqueta Real')
    plt.xlabel('Predicción')
    plt.tight_layout()
    plt.savefig('data/if_confusion_matrix.png', dpi=150)
    plt.show()

---

## Modelo 2: Autoencoder Neuronal - Detección por Reconstrucción

Un **autoencoder** es una red neuronal que aprende a reconstruir datos normales.
Cuando se le presenta tráfico anómalo, el error de reconstrucción es alto,
permitiendo distinguir anomalías por su patrón diferente.


In [ ]:
# Visualización 1: Detección a lo largo de la serie temporal
normal_idx = np.where(predictions == 1)[0]
anomaly_idx = np.where(predictions == -1)[0]

plt.figure(figsize=(14, 5))

# Gráfico 1: bytes_sent
plt.subplot(1, 2, 1)
plt.scatter(normal_idx, df.iloc[normal_idx]['bytes_sent'], 
           s=10, c='steelblue', alpha=0.5, label='Normal')
plt.scatter(anomaly_idx, df.iloc[anomaly_idx]['bytes_sent'], 
           s=30, c='red', alpha=0.8, marker='X', label='Anomalía (IF)')
plt.xlabel('Índice de muestra')
plt.ylabel('bytes_sent')
plt.title('Bytes enviados: Detección de anomalías')
plt.legend()
plt.grid(True, alpha=0.3)

# Gráfico 2: Puntuaciones de anomalía
plt.subplot(1, 2, 2)
plt.hist(anomaly_scores[predictions == 1], bins=50, color='steelblue', alpha=0.6, label='Normal')
plt.hist(anomaly_scores[predictions == -1], bins=50, color='red', alpha=0.6, label='Anomalía')
plt.xlabel('Anomaly Score')
plt.ylabel('Frecuencia')
plt.title('Distribución de puntuaciones de anomalía')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/if_visualization.png', dpi=150)
plt.show()

print("✓ Gráficos guardados en data/if_visualization.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report

print("=" * 60)
print("AUTOENCODER NEURONAL - DETECCIÓN DE INTRUSIONES")
print("=" * 60)

# ─ Preparar datos ─────────────────────────────────────────────────────
df = pd.read_csv('data/network_traffic.csv').dropna()

# Codificar protocolo
le = LabelEncoder()
if df['protocol'].dtype == object:
    df['protocol'] = le.fit_transform(df['protocol'])

feature_cols = ['bytes_sent', 'bytes_recv', 'duration', 'src_port', 'dst_port', 'protocol']
scaler = MinMaxScaler()
X_all = scaler.fit_transform(df[feature_cols])
y_all = df['label'].values

# ─ Separación train/test ──────────────────────────────────────────────
# El autoencoder se entrena SOLO con tráfico normal para aprender la normalidad
X_normal = X_all[y_all == 0]

X_train, X_val = train_test_split(X_normal, test_size=0.2, random_state=42)
X_test = X_all
y_test = y_all

print(f"\n[Dataset split]")
print(f"  Train (solo normales): {X_train.shape[0]}")
print(f"  Val (solo normales):   {X_val.shape[0]}")
print(f"  Test (normal+anomalo): {X_test.shape[0]}")
print(f"    - Normal: {(y_test == 0).sum()}")
print(f"    - Anómalo: {(y_test == 1).sum()}")

# ─ Arquitectura del autoencoder ────────────────────────────────────────
input_dim = X_train.shape[1]

def construir_autoencoder(input_dim, dim_latente=8):
    """
    Autoencoder simétrico con botella de información (latent layer).
    
    Encoder: input(6) -> 16 -> 8 (latent)
    Decoder: 8 -> 16 -> output(6)
    """
    entrada = keras.Input(shape=(input_dim,))
    
    # Encoder
    x = layers.Dense(16, activation='relu')(entrada)
    x = layers.Dense(dim_latente, activation='relu')(x)
    
    # Decoder
    x = layers.Dense(16, activation='relu')(x)
    salida = layers.Dense(input_dim, activation='sigmoid')(x)
    
    autoencoder = keras.Model(entrada, salida, name='autoencoder')
    return autoencoder

ae_model = construir_autoencoder(input_dim)
ae_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

print(f"\n[Arquitectura del Autoencoder]")
ae_model.summary()

In [ ]:
# ─ Entrenamiento ──────────────────────────────────────────────────────
print("\n[Iniciando entrenamiento...]")
hist = ae_model.fit(
    X_train, X_train,                           # entrada = salida (reconstrucción)
    epochs=50,
    batch_size=64,
    validation_data=(X_val, X_val),
    verbose=1,
    shuffle=True
)

# ─ Calcular error de reconstrucción ────────────────────────────────────
print("\n[Predicción en conjunto de prueba...]")
X_test_reconstructed = ae_model.predict(X_test, verbose=0)

# MSE (Mean Squared Error) como métrica de anomalía
reconstruction_errors = np.mean(np.power(X_test - X_test_reconstructed, 2), axis=1)

# ─ Determinar umbral dinámico ──────────────────────────────────────────
# Usamos el percentil 95 del error en datos de entrenamiento
X_train_reconstructed = ae_model.predict(X_train, verbose=0)
train_errors = np.mean(np.power(X_train - X_train_reconstructed, 2), axis=1)
threshold_ae = np.percentile(train_errors, 95)

print(f"\n[Estadísticas de error de reconstrucción]")
print(f"  Error promedio en train: {train_errors.mean():.6f}")
print(f"  Percentil 95 (umbral): {threshold_ae:.6f}")
print(f"  Error máximo en test: {reconstruction_errors.max():.6f}")

# ─ Clasificación de anomalías ──────────────────────────────────────────
ae_predictions = (reconstruction_errors > threshold_ae).astype(int)

print(f"\n[Resultados del Autoencoder]")
print(f"  Anomalías detectadas: {ae_predictions.sum()}")
print(f"  Muestras normales: {(ae_predictions == 0).sum()}")

# ─ Evaluación ──────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("EVALUACIÓN - AUTOENCODER")
print("=" * 60)
print(classification_report(y_test, ae_predictions, target_names=['Normal', 'Anomalía']))

In [ ]:
# Visualización 1: Curva de aprendizaje
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pérdida de entrenamiento
axes[0].plot(hist.history['loss'], label='Pérdida Train', linewidth=2)
axes[0].plot(hist.history['val_loss'], label='Pérdida Val', linewidth=2)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('MSE')
axes[0].set_title('Curva de Aprendizaje del Autoencoder')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE de entrenamiento
axes[1].plot(hist.history['mae'], label='MAE Train', linewidth=2)
axes[1].plot(hist.history['val_mae'], label='MAE Val', linewidth=2)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('MAE')
axes[1].set_title('Error Absoluto Medio')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/autoencoder_training.png', dpi=150)
plt.show()

# Visualización 2: Distribución del error de reconstrucción
plt.figure(figsize=(10, 5))

normal_errors = reconstruction_errors[y_test == 0]
anomaly_errors = reconstruction_errors[y_test == 1]

plt.hist(normal_errors, bins=60, alpha=0.6, color='steelblue', label='Normal')
plt.hist(anomaly_errors, bins=30, alpha=0.8, color='red', label='Anomalía')
plt.axvline(threshold_ae, color='green', linestyle='--', linewidth=2, label=f'Umbral: {threshold_ae:.5f}')
plt.xlabel('Error de Reconstrucción (MSE)')
plt.ylabel('Frecuencia')
plt.title('Distribución del Error de Reconstrucción')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('data/autoencoder_reconstruction_error.png', dpi=150)
plt.show()

print("✓ Gráficos guardados: data/autoencoder_training.png, data/autoencoder_reconstruction_error.png")

---

## Persistencia de Artefactos

Se guardan los modelos entrenados, scaler y dataset con predicciones
para reutilización en siguientes análisis.


In [ ]:
import joblib
import os

# Crear directorio de modelos si no existe
os.makedirs('models', exist_ok=True)

# ─ Guardar modelo Isolation Forest ─────────────────────────────────────
joblib.dump(iso_forest, 'models/isolation_forest_model.pkl')
print("[+] Isolation Forest guardado: models/isolation_forest_model.pkl")

# ─ Guardar Autoencoder ─────────────────────────────────────────────────
ae_model.save('models/autoencoder_detector.keras')
print("[+] Autoencoder guardado: models/autoencoder_detector.keras")

# ─ Guardar scaler ──────────────────────────────────────────────────────
joblib.dump(scaler, 'models/network_feature_scaler.pkl')
print("[+] Scaler guardado: models/network_feature_scaler.pkl")

# ─ Guardar encoder label para protocolo ────────────────────────────────
joblib.dump(le, 'models/protocol_encoder.pkl')
print("[+] Protocol encoder guardado: models/protocol_encoder.pkl")

# ─ Crear dataset con todas las predicciones ────────────────────────────
df_results = pd.read_csv('data/network_traffic.csv')
df_results['IF_anomaly'] = (iso_forest.predict(X_scaled) == -1).astype(int)
df_results['IF_score'] = iso_forest.decision_function(X_scaled)
df_results['AE_anomaly'] = ae_predictions
df_results['AE_error'] = reconstruction_errors

df_results.to_csv('data/network_traffic_with_predictions.csv', index=False)
print("[+] Dataset con predicciones guardado: data/network_traffic_with_predictions.csv")

print("\n" + "=" * 60)
print("✓ PIPELINE COMPLETADO Y PERSISTIDO")
print("=" * 60)
print("\nArchivos generados:")
print("  Modelos: models/isolation_forest_model.pkl, models/autoencoder_detector.keras")
print("  Datos: data/network_traffic.csv, data/network_traffic_with_predictions.csv")
print("  Gráficos: data/*.png")
print("\nPróximo paso: Ejecutar 03_deteccion_malware.ipynb")